Proof of Concept to extract the components of the ARC SFL Region Weather Sitrep consisting of:

NWS reports for Miami, Tampa and Melbourne. Main image and description
NHC Seven-Day Graphical Tropical Weather Outlook
Florida Department of Agriculture Fire Danger Maps
This will create a word document that can be used as the source for cut an paste into the template. Hopefully, one day this will replace the template but for now this should help

Looks like the NWS pages are static so it should be easy to just parse the data from the page.

The NHC page must be rendered to be able to obtain the information

In [ ]:
# Pre requisites
%pip install requests
%pip install beautifulsoup4
%pip install python-docx
%pip install selenium
%pip install Pillow

In [ ]:
import requests
import io
import re
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
from docx import Document
from docx.shared import Inches, Pt
from docx.enum.section import WD_SECTION
from PIL import Image, ImageDraw, ImageFont

In [ ]:
# Constants and parameters

nws_forecast_offices = {
    'NWS Miami Office': 'mfl',
    'NWS Tampa Office': 'tbw',
    'NWS Melbourne Office': 'mlb'
    }
nws_base_url = 'https://weather.gov'
# nws_image_selector = 'div.graphicast img'
nws_description_selector = '.graphicast > div.description'

# create dictionaries to store an image and description for each of the keys
nws_image = {}
nws_description = {}

nhc_url = 'https://www.nhc.noaa.gov/'
nhc_7day_img_url = 'https://www.nhc.noaa.gov/xgtwo/two_atl_7d0.png'

fire_danger_image_url = 'https://weather.fdacs.gov/FDI/images/FL-latest-fcst.png'

output_file_name = './drafts/sitrep_workfile'

In [ ]:

def request_page(url):
  """
  fetch a page and return the text in the response
  """
  try:
    response = requests.get(url)
    response.raise_for_status()
    return response.text
  except requests.exceptions.RequestException as e:
        print(f"Error fetching URL: {e}")
  except Exception as e:
        print(f"An error occurred: {e}")

In [ ]:
def extract_nws_info(nws_response):
  """
  Obtain the description from the bottom of the first image on the response and the URL of the image
  """
  soup = BeautifulSoup(nws_response, 'html.parser')
  desc = soup.select_one(nws_description_selector).text.strip().replace('Click/tap image to enlarge | ','')
  
  # Check if a description is provided
  if len(desc) == 0:
    desc = "*** DESCRIPTION NOT PROVIDED ON THIS PAGE ***" 
  
  # get the url of the first image 'in the page'
  img_url = soup.select_one('div.graphicast img').attrs['src']

  return { "description": desc, "image_url": img_url }

In [ ]:
def show_image_error(file_path='./no-image.png'):
  """
  Create a png image with an error message
  """
  image = Image.open(file_path)
  
  img_byte_arr = io.BytesIO()
  image.save(img_byte_arr,format='PNG')

  return img_byte_arr


In [ ]:
def get_image(img_url):
  """
  fetch an image and return it as stream
  """
  print(f"fetching image from: '{img_url}'")
  try:
    img_response = requests.get(img_url)
    img_response.raise_for_status()
    image_stream = io.BytesIO(img_response.content)
  except:
    image_stream = show_image_error()
  return image_stream

In [ ]:
from datetime import datetime

def append_datetime(input_string):
  """
  Appends the current datetime in yymmdd_hhmmss format to a string.

  Args:
    input_string: The string to which the datetime will be appended.

  Returns:
    The string with the datetime appended.
  """
  short_fmt = "_%m%d_%H%M"
  long_fmt = "_%Y%m%d_%H%M%S"
  now = datetime.now()
  datetime_string = now.strftime(short_fmt)
  return input_string + datetime_string
     

In [ ]:
def retrieve_nws_info():
  """
  Populate the dictionaries with the images and descriptions from the NWS pages
  """
  for e in nws_forecast_offices:
    print(f"Processing: {e}")
    page_url = f"{nws_base_url}/{nws_forecast_offices[e]}/"
    nws_response = request_page(page_url)
    
    nws_info = extract_nws_info(nws_response)
    nws_description[e] = nws_info["description"] 
    image_url = nws_info["image_url"] 
    nws_image[e] = get_image(image_url)
  
     

In [ ]:
def extract_text_arrays_from_url(url):
    """
    Extract description of items in NHC Seven-day Graphical Tropical Weather Outlook
    """
    nhc_text = []
    try:
        # Set up Selenium with headless Chrome
        chrome_options = Options()
        chrome_options.add_argument("--headless")
        chrome_options.add_argument("--disable-gpu")
        chrome_options.add_argument("--no-sandbox")
        chrome_options.add_argument("--disable-dev-shm-usage")
        driver = webdriver.Chrome(options=chrome_options)

        # Fetch the fully loaded page
        driver.get(url)
        page_source = driver.page_source
        driver.quit()

        # Parse the HTML content with BeautifulSoup
        soup = BeautifulSoup(page_source, 'html.parser')

        # Find all <script> tags
        script_tags = soup.find_all('script')

        # Regex pattern to find arrays named 'Text'
        # Matches: Text[0]=[...], Text[1]=[...], etc.
        pattern = r'Text\[\d+\]\s*=\s*\[([^\]]*)\]'

        found_arrays = []
        for script in script_tags:
            if script.string:  # Check if script tag has content
                # Find all matches in the JavaScript content
                matches = re.finditer(pattern, script.string, re.MULTILINE)
                found_arrays.extend([match.group(1).strip() for match in matches])

        if found_arrays:
            print("Found JavaScript arrays named 'Text' (HTML tags and '(click for details)' removed, each sub-item on a new line):")
            for idx, content in enumerate(found_arrays, 1):
                # Split the array content by commas, accounting for quoted strings
                items = re.split(r',\s*(?=(?:[^"]*"[^"]*")*[^"]*$)', content)
                clean_items = []
                for item in items:
                    # Remove surrounding quotes if present
                    item = item.strip().strip("'").strip('"')
                    if item:
                        # Parse HTML
                        soup = BeautifulSoup(item, 'html.parser')
                        # Get all text content, splitting by <br> tags
                        sub_items = []
                        for element in soup.find_all(text=True, recursive=True):
                            text = element.strip()
                            if text:
                                # Split by newlines or spaces if necessary
                                sub_texts = text.split('\n')
                                for sub_text in sub_texts:
                                    sub_text = sub_text.strip()
                                    if sub_text:
                                        # Remove "(click for details)" from the sub-text
                                        sub_text = sub_text.replace('(click for details)', '').strip()
                                        if sub_text:  # Only add non-empty sub-text
                                            sub_items.append(sub_text)
                        # Add sub-items to clean_items
                        clean_items.extend(sub_items)
                # Join all sub-items with newlines to form a paragraph
                paragraph = '\n'.join(clean_items).strip()
                # print(f"\nText[{idx-1}] Paragraph:")
                # print(paragraph)
                nhc_text.append(paragraph)
        else:
            print("No JavaScript arrays named 'Text' found in the webpage.")

    except Exception as e:
        print(f"An error occurred: {e}")
    print("----")
    return nhc_text

In [ ]:
# prepare document
document = Document()
font = document.styles['Normal'].font
font.name = 'Calibri'
font.size = Pt(12)
document.add_heading('ARC SFL Region Weather Sitrep workfile', 0)

# NWS offices
retrieve_nws_info()
for ofc in nws_forecast_offices:
  document.add_heading(ofc, level=2)
  document.add_picture(nws_image[ofc], width=Inches(5.5))

  document.add_paragraph(nws_description[ofc].strip())
  document.add_section(WD_SECTION.NEW_PAGE)

# NHC 7-day forecast
nhc_img = get_image(nhc_7day_img_url)
document.add_heading('Seven Day Forecast', level=2)
document.add_picture(nhc_img, width=Inches(5.5))
for txt in extract_text_arrays_from_url(nhc_url):
  document.add_paragraph(txt)
document.add_section(WD_SECTION.NEW_PAGE)

# Fire Risk
print(fire_danger_image_url)
fire_risk_img = get_image(fire_danger_image_url)
document.add_heading('Fire Danger Maps', level=2)
document.add_picture(fire_risk_img, width=Inches(4.0))
# uncomment if another section is added to the report
# document.add_section(WD_SECTION.NEW_PAGE)


# Save document
document.save(append_datetime(output_file_name)+'.docx')